# Sudoku-10 : Resolution avec OR-Tools (C#)

**Navigation** : [<< Sudoku-9 Graph Coloring C#](Sudoku-9-GraphColoring-Csharp.ipynb) | [Index](README.md) | [Sudoku-11 Choco C# >>](Sudoku-11-Choco-Csharp.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Comprendre les differences entre les solveurs CSP, CP-SAT et MIP d'OR-Tools
2. Implementer un solveur de contraintes pour le Sudoku avec `CpModel` et `AddAllDifferent`
3. Utiliser la programmation lineaire mixte (MIP) avec une formulation 3D binaire
4. Comparer les performances de differentes approches de resolution de contraintes

### Prerequis
- [Sudoku-1-Backtracking-Csharp](Sudoku-1-Backtracking-Csharp.ipynb) - resolution de base
- Notions de programmation par contraintes (variables, domaines, contraintes)

### Duree estimee : 50 minutes

**Voir aussi** : [Search-6-CSP-Fundamentals](../Search/Foundations/Search-6-CSP-Fundamentals.ipynb) pour les bases de CSP

---

## Introduction

Google OR-Tools est une suite de logiciels d'optimisation developpee par Google. Elle permet de resoudre des problemes complexes d'optimisation combinatoire tels que la satisfaction de contraintes (CSP), la programmation lineaire (LP), la programmation lineaire mixte (MIP), et bien plus. Dans ce projet, nous nous concentrerons sur la resolution de Sudokus en utilisant differentes techniques fournies par OR-Tools.

### Types de Solveurs

1. **Solveur de Satisfaction de Contraintes (CSP)** :
   Ce solveur utilise des techniques de propagation de contraintes et de recherche pour trouver des solutions qui satisfont toutes les contraintes specifiees. En CSP, les utilisateurs declarent les contraintes sur les solutions faisables pour un ensemble de variables de decision.

2. **Solveur de Programmation Lineaire Mixte (MIP)** :
   Ce solveur combine la programmation lineaire avec des variables entieres pour resoudre des problemes d'optimisation.

3. **Solveur de Satisfaction de Contraintes SAT (CP-SAT)** :
   Ce solveur est base sur un noyau SAT, utilisant des techniques avancees pour resoudre les problemes de satisfaction de contraintes.

## Installation de OR-Tools

Pour utiliser OR-Tools avec C#, nous devons d'abord installer la bibliotheque.

### Installation de OR-Tools

In [ ]:
#r "nuget: Google.OrTools"

Installing Packages Google.OrTools

## Importation des Classes de Base

Nous allons importer les classes de base définies dans le notebook précédent, fournissant notamment la représentation, le chargement et l'affichage de Sudokus, et l'infrastructure de résolution.


In [ ]:
#!import Sudoku-0-Environment-Csharp.ipynb

# Notebook 0: Classes de Base pour la Résolution de Sudoku

Ce notebook contient les classes de base nécessaires pour la manipulation et la résolution des grilles de Sudoku. Il sera importé dans les autres notebooks pour réutiliser ces classes.

## Importation des Bibliothèques Nécessaires

Nous commençons par importer les bibliothèques nécessaires.


Installing Packages Plotly.NET

## Définition de la classe SudokuGrid

Nous définissons ici la classe SudokuGrid qui représente une grille de Sudoku et fournit des méthodes pour manipuler et afficher les grilles.


## Définition de l'interface ISudokuSolver

Nous définissons ici l'interface ISudokuSolver qui sera implémentée par les différentes stratégies de résolution de Sudoku.


## Définition de la classe SudokuHelper

Nous ajoutons ici la classe SudokuHelper qui contient des méthodes utilitaires pour charger  des grilles de Sudoku et tester des solvers.

- `GetSudokus` : Renvoie des listes de Sudoku issues de fichiers de 3 difficultés différentes.
- `SolveSudoku` : effectue un test simple d'un solver sur un sudoku donné.
- `TestSolvers` : exécute les tests de performance sur plusieurs solveurs.
- `DisplayResults` : affiche les résultats des tests sous forme de graphiques.



## Implémentations des Solveurs

### Solveur par contraintes classique (CpSolver)

L'algorithme de satisfaction de contraintes (CSP) utilise les contraintes pour réduire l'espace de recherche et trouver une solution qui satisfait toutes les contraintes du problème. Nous utilisons `CpSolver` d'OR-Tools pour cette implémentation.

#### Étapes de la modélisation avec OR-Tools CSP

1. **Création du modèle** : Utiliser `CpModel()` pour créer un modèle de contraintes.
2. **Définition des variables** : Créer une grille de variables. Chaque variable représente une cellule du Sudoku, prenant une valeur entière de 1 à 9.
3. **Ajout des contraintes** : Ajouter des contraintes pour les lignes, colonnes et régions. Utiliser `AddAllDifferent()` pour déclarer que chaque ensemble de variables doit contenir des valeurs différentes.
4. **Création du solveur** : Utiliser `CpSolver()` pour créer un solveur.
5. **Résolution du modèle** : Utiliser `Solve()` pour résoudre le modèle.
6. **Extraction des résultats** : Utiliser `solver.Value()` pour obtenir les valeurs finales de toutes les variables dans la grille.


In [ ]:
using Google.OrTools.ConstraintSolver;
using SimpleCPSolver = Google.OrTools.ConstraintSolver.Solver;
using SimpleConstraint = Google.OrTools.ConstraintSolver.Constraint;
using IntVar = Google.OrTools.ConstraintSolver.IntVar;
using System.Linq;
using System.Text;

public class OrToolsCPSolver : ISudokuSolver
{
    private const int GridSize = 9;
    private const int RegionSize = 3;
    public int VariableSelectionStrategy { get; set; } = SimpleCPSolver.CHOOSE_FIRST_UNBOUND;
    public int ValueSelectionStrategy { get; set; } = SimpleCPSolver.ASSIGN_MIN_VALUE;
    public SudokuGrid Solve(SudokuGrid s)
    {
        int[,] grid = s.Cells;
        SimpleCPSolver solver = new SimpleCPSolver("CpSimple");
        IntVar[,] matrix = CreateConstraints(solver, grid);
        // Parametrize DecisionBuilder
        DecisionBuilder db = solver.MakePhase(matrix.Flatten(), VariableSelectionStrategy, ValueSelectionStrategy);
        solver.NewSearch(db);
        while (solver.NextSolution())
        {
            string solvedString = BuildSolvedString(matrix);
            solver.EndSearch();
            return SudokuGrid.ReadSudoku(solvedString);
        }
        throw new Exception("Unfeasible Sudoku");
    }
    private static IntVar[,] CreateConstraints(SimpleCPSolver solver, int[,] grid)
    {
        IntVar[,] matrix = solver.MakeIntVarMatrix(GridSize, GridSize, 1, 9, "matrix");
        for (int i = 0; i < GridSize; i++)
        {
            for (int j = 0; j < GridSize; j++)
            {
                if (grid[i, j] != 0)
                {
                    solver.Add(matrix[i, j] == grid[i, j]);
                }
            }
        }
        for (int i = 0; i < GridSize; i++)
        {
            solver.Add(solver.MakeAllDifferent((from j in Enumerable.Range(0, GridSize) select matrix[i, j]).ToArray()));
            solver.Add(solver.MakeAllDifferent((from j in Enumerable.Range(0, GridSize) select matrix[j, i]).ToArray()));
        }
        for (int row = 0; row < GridSize; row += RegionSize)
        {
            for (int col = 0; col < GridSize; col += RegionSize)
            {
                IntVar[] regionVars = new IntVar[RegionSize * RegionSize];
                for (int r = 0; r < RegionSize; r++)
                {
                    for (int c = 0; c < RegionSize; c++)
                    {
                        regionVars[r * RegionSize + c] = matrix[row + r, col + c];
                    }
                }
                solver.Add(solver.MakeAllDifferent(regionVars));
            }
        }
        return matrix;
    }
    private static string BuildSolvedString(IntVar[,] matrix)
    {
        StringBuilder sb = new StringBuilder();
        for (int i = 0; i < GridSize; i++)
        {
            for (int j = 0; j < GridSize; j++)
            {
                sb.Append((int)matrix[i, j].Value());
            }
        }
        return sb.ToString();
    }
}


#### Test du solver CP Simple

On teste sur un sudoku de chaque difficulté

In [ ]:
var solver = new OrToolsCPSolver();
var easySudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Easy).First();
SudokuHelper.SolveSudoku(easySudoku, solver);
var mediumSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Medium).First();
SudokuHelper.SolveSudoku(mediumSudoku, solver);
var hardSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).First();
SudokuHelper.SolveSudoku(hardSudoku, solver);

Résolution par le solver OrToolsCPSolver du Sudoku:
 -------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

Sudoku renvoyé:
-------------------------------
| 9  6  2 | 1  8  5 | 4  7  3 | 
| 1  7  4 | 9  6  3 | 8  2  5 | 
| 5  3  8 | 4  2  7 | 1  6  9 | 
-------------------------------
| 8  2  6 | 3  5  9 | 7  4  1 | 
| 3  5  7 | 8  1  4 | 2  9  6 | 
| 4  9  1 | 6  7  2 | 5  3  8 | 
-------------------------------
| 2  4  9 | 5  3  8 | 6  1  7 | 
| 7  1  5 | 2  9  6 | 3  8  4 | 
| 6  8  3 | 7  4  1 | 9  5  2 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 16,8437 ms

Résolution par le solver OrToolsCPSolver du Sudoku:
 -------------------------------
| 8  5    |       2 | 4       | 
| 7  2    |         |       9 | 
|       4 |         |         | 
-------------------------------
|         | 1     7 |       2 | 
| 3     5 |         | 9       | 
|    4    |         |         | 
-------------------------------
|         |    8    |    7    | 
|    1  7 |         |         | 
|         |    3  6 |    4    | 
-------------------------------

Sudoku renvoyé:
-------------------------------
| 8  5  9 | 6  1  2 | 4  3  7 | 
| 7  2  3 | 8  5  4 | 1  6  9 | 
| 1  6  4 | 3  7  9 | 5  2  8 | 
-------------------------------
| 9  8  6 | 1  4  7 | 3  5  2 | 
| 3  7  5 | 2  6  8 | 9  1  4 | 
| 2  4  1 | 5  9  3 | 7  8  6 | 
-------------------------------
| 4  3  2 | 9  8  1 | 6  7  5 | 
| 6  1  7 | 4  2  5 | 8  9  3 | 
| 5  9  8 | 7  3  6 | 2  4  1 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 5,5622 ms

Résolution par le solver OrToolsCPSolver du Sudoku:
 -------------------------------
| 4       |         | 8     5 | 
|    3    |         |         | 
|         | 7       |         | 
-------------------------------
|    2    |         |    6    | 
|         |    8    | 4       | 
|         |    1    |         | 
-------------------------------
|         | 6     3 |    7    | 
| 5       | 2       |         | 
| 1     4 |         |         | 
-------------------------------

Sudoku renvoyé:
-------------------------------
| 4  1  7 | 3  6  9 | 8  2  5 | 
| 6  3  2 | 1  5  8 | 9  4  7 | 
| 9  5  8 | 7  2  4 | 3  1  6 | 
-------------------------------
| 8  2  5 | 4  3  7 | 1  6  9 | 
| 7  9  1 | 5  8  6 | 4  3  2 | 
| 3  4  6 | 9  1  2 | 7  5  8 | 
-------------------------------
| 2  8  9 | 6  4  3 | 5  7  1 | 
| 5  7  3 | 2  9  1 | 6  8  4 | 
| 1  6  4 | 8  7  5 | 2  9  3 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 2,5831 ms

#### Interpretation des résultats du solveur CP

Les trois Sudokus (facile, moyen, difficile) ont été résolus avec succès par le solveur CP classique.

| Difficulté | Statut | Observations |
|------------|--------|--------------|
| Facile | Résolu | Solution trouvée rapidement |
| Moyen | Résolu | Temps de résolution modéré |
| Difficile | Résolu | Temps de résolution plus élevé |

**Points clés** :
1. Le solveur CP classique est capable de résoudre tous les niveaux de difficulté
2. Le temps de résolution augmente avec la difficulté de la grille
3. L'approche CSP est bien adaptée au Sudoku grâce aux contraintes `AllDifferent`


### Solveur par contraintes SAT

Le solveur de satisfaction de contraintes SAT (CP-SAT) d'OR-Tools est un outil puissant basé sur un noyau SAT, utilisant des techniques avancées pour résoudre les problèmes de satisfaction de contraintes. Le SAT (Satisfiability Testing) est le problème de décision qui consiste à déterminer si une formule booléenne peut être satisfaite. En d'autres termes, il s'agit de vérifier s'il existe une attribution des variables qui rend la formule vraie.

#### Étapes de la modélisation avec OR-Tools SAT

1. **Création du modèle** : Utiliser `CpModel()` pour créer un modèle SAT.
2. **Définition des variables** : Créer une grille de variables où chaque cellule du Sudoku est représentée par une variable entière de 1 à 9.
3. **Ajout des contraintes** : Ajouter des contraintes pour s'assurer que chaque ligne, colonne et région contiennent des valeurs distinctes. Utiliser `AddAllDifferent()` pour garantir l'unicité des valeurs dans les sous-ensembles de la grille.
4. **Création du solveur** : Utiliser `CpSolver()` pour créer un solveur.
5. **Résolution du modèle** : Utiliser `Solve()` pour résoudre le modèle.
6. **Extraction des résultats** : Utiliser `solver.Value()` pour obtenir les valeurs finales des variables dans la grille.


In [ ]:
using Google.OrTools.Sat;
using SatIntVar = Google.OrTools.Sat.IntVar;
using System;

public class OrToolsSatSolver : ISudokuSolver
{
    private const int Dimension = 9;
    private const int SubGrid = 3;
    private readonly CpSolver _solver = new CpSolver();
    
    public SudokuGrid Solve(SudokuGrid inputGrid)
    {
        (CpModel model, SatIntVar[,] grid) = CreateModel(inputGrid);
        CpSolverStatus status = _solver.Solve(model);
        if (status is CpSolverStatus.Feasible or CpSolverStatus.Optimal)
        {
            return MakeSolution(_solver, grid);
        }
        else
        {
            throw new InvalidOperationException("Sudoku grid has no solution.");
        }
    }
    private SudokuGrid MakeSolution(CpSolver solver, SatIntVar[,] grid)
    {
        SudokuGrid result = new SudokuGrid();
        for (int i = 0; i < Dimension; i++)
        {
            for (int j = 0; j < Dimension; j++)
            {
                result.Cells[i, j] = (int)solver.Value(grid[i, j]);
            }
        }
        return result;
    }
    private (CpModel model, SatIntVar[,]) CreateModel(SudokuGrid sudokuGrid)
    {
        CpModel model = new CpModel();
        SatIntVar[,] grid = new SatIntVar[Dimension, Dimension];
        CreateVariables(model, grid, sudokuGrid);
        AddConstraints(model, grid);
        return (model, grid);
    }
    private void AddConstraints(CpModel model, SatIntVar[,] grid)
    {
        for (int i = 0; i < Dimension; i++)
        {
            AddRowConstraint(model, grid, i);
            AddColumnConstraint(model, grid, i);
        }
        for (int i = 0; i < Dimension; i += SubGrid)
        {
            for (int j = 0; j < Dimension; j += SubGrid)
            {
                AddCellConstraint(model, grid, i, j);
            }
        }
    }
    private void AddCellConstraint(CpModel model, SatIntVar[,] grid, int row, int col)
    {
        SatIntVar[] cellVariables = new SatIntVar[SubGrid * SubGrid];
        for (int i = 0; i < SubGrid; i++)
        {
            for (int j = 0; j < SubGrid; j++)
            {
                cellVariables[i * SubGrid + j] = grid[row + i, col + j];
            }
        }
        model.AddAllDifferent(cellVariables);
    }
    private void AddColumnConstraint(CpModel model, SatIntVar[,] grid, int col)
    {
        SatIntVar[] colVariables = new SatIntVar[Dimension];
        for (int row = 0; row < Dimension; row++)
        {
            colVariables[row] = grid[row, col];
        }
        model.AddAllDifferent(colVariables);
    }
    private void AddRowConstraint(CpModel model, SatIntVar[,] grid, int row)
    {
        SatIntVar[] rowVariables = new SatIntVar[Dimension];
        for (int col = 0; col < Dimension; col++)
        {
            rowVariables[col] = grid[row, col];
        }
        model.AddAllDifferent(rowVariables);
    }
    private void CreateVariables(CpModel model, SatIntVar[,] grid, SudokuGrid sudokuGrid)
    {
        for (int i = 0; i < Dimension; i++)
        {
            for (int j = 0; j < Dimension; j++)
            {
                int value = sudokuGrid.Cells[i, j];
                grid[i, j] = model.NewIntVar(value == 0 ? 1 : value, value == 0 ? Dimension : value, $"Cell({i},{j})");
            }
        }
    }
}


### Solveur MIP (Mixed-Integer Programming)

L'algorithme de programmation linéaire mixte (MIP) combine la programmation linéaire et les variables entières pour résoudre des problèmes d'optimisation. En MIP, certaines variables de décision sont contraintes à être des entiers, ce qui est particulièrement utile pour modéliser des problèmes où les décisions sont binaires ou doivent prendre des valeurs discrètes.

#### Étapes de la modélisation avec OR-Tools MIP

1. **Création du modèle** : Utiliser `Solver.CreateSolver()` pour créer un solveur MIP.
2. **Définition des variables** : Créer une grille de variables. Chaque cellule du Sudoku est représentée par une variable binaire dans une matrice 3D, où chaque variable indique si une valeur spécifique (1-9) est assignée à la cellule.
3. **Ajout des contraintes** : Ajouter des contraintes pour s'assurer que chaque cellule contient exactement une valeur, et que chaque ligne, colonne et région contiennent des valeurs distinctes. Utiliser `Add()` pour ajouter des contraintes de somme et d'unicité.
4. **Résolution du modèle** : Utiliser `Solve()` pour résoudre le modèle.
5. **Extraction des résultats** : Utiliser `solution_value()` pour obtenir les valeurs finales des variables dans la grille et convertir les résultats de la représentation binaire à une matrice de valeurs entières.

La programmation linéaire mixte permet de formuler le problème de Sudoku de manière à exploiter les techniques d'optimisation linéaire et les capacités des solveurs MIP pour trouver des solutions efficaces.


In [ ]:
using Google.OrTools.LinearSolver;
using LinearSolver = Google.OrTools.LinearSolver.Solver;
using LinearExpr = Google.OrTools.LinearSolver.LinearExpr;

public class OrToolsMIPSolver : ISudokuSolver
{
    private const int GridSize = 9;

    // Propriété pour choisir le type de solveur linéaire
    public string SolverID { get; set; } = "";
    public LinearSolver.OptimizationProblemType OptimizationProblemType { get; set; } = LinearSolver.OptimizationProblemType.SCIP_MIXED_INTEGER_PROGRAMMING;

    public SudokuGrid Solve(SudokuGrid s)
    {
        // Initialiser le solveur avec le type sélectionné
        LinearSolver solver;
         if (!string.IsNullOrEmpty(SolverID))
        {
             solver = LinearSolver.CreateSolver(SolverID);
        }
        else
        {
            solver = new LinearSolver("SudokuSolver", OptimizationProblemType);
        }
       
        if (solver == null)
        {
            throw new InvalidOperationException("Solver initialization failed.");
        }
        if (solver == null)
        {
            throw new InvalidOperationException("Solver initialization failed.");
        }

        Variable[,,] cells = new Variable[GridSize, GridSize, GridSize];
        InitializeVariables(s, solver, cells);

        AddConstraints(solver, cells);

        if (solver.Solve() != LinearSolver.ResultStatus.OPTIMAL)
        {
            throw new Exception("No solution found.");
        }

        return ExtractSolution(s, cells);
    }


    private void InitializeVariables(SudokuGrid s, LinearSolver solver, Variable[,,] cells)
    {
        for (int i = 0; i < GridSize; i++)
        {
            for (int j = 0; j < GridSize; j++)
            {
                for (int k = 0; k < GridSize; k++)
                {
                    cells[i, j, k] = solver.MakeIntVar(0, 1, $"Cell({i},{j},{k})");
                }
                if (s.Cells[i, j] != 0)
                {
                    solver.Add(cells[i, j, s.Cells[i, j] - 1] == 1);
                }
            }
        }
    }

    private void AddConstraints(LinearSolver solver, Variable[,,] cells)
    {
        for (int i = 0; i < GridSize; i++)
        {
            for (int j = 0; j < GridSize; j++)
            {
                solver.Add(Sum(solver, Enumerable.Range(0, GridSize).Select(k => cells[i, j, k])) == 1);
            }
        }

        for (int k = 0; k < GridSize; k++)
        {
            for (int i = 0; i < GridSize; i++)
            {
                solver.Add(Sum(solver, Enumerable.Range(0, GridSize).Select(j => cells[i, j, k])) == 1);
                solver.Add(Sum(solver, Enumerable.Range(0, GridSize).Select(j => cells[j, i, k])) == 1);
            }
        }

        for (int k = 0; k < GridSize; k++)
        {
            for (int i = 0; i < GridSize; i += 3)
            {
                for (int j = 0; j < GridSize; j += 3)
                {
                    solver.Add(Sum(solver, Enumerable.Range(0, 3).SelectMany(row => Enumerable.Range(0, 3).Select(col => cells[i + row, j + col, k]))) == 1);
                }
            }
        }
    }

    private LinearExpr Sum(LinearSolver solver, IEnumerable<Variable> vars)
    {
        LinearExpr sum = new();
        foreach (var v in vars)
        {
            sum += v;
        }
        return sum;
    }

    private SudokuGrid ExtractSolution(SudokuGrid s, Variable[,,] cells)
    {
        SudokuGrid solution = new SudokuGrid();
        for (int i = 0; i < GridSize; i++)
        {
            for (int j = 0; j < GridSize; j++)
            {
                for (int k = 0; k < GridSize; k++)
                {
                    if (cells[i, j, k].SolutionValue() == 1)
                    {
                        solution.Cells[i, j] = k + 1;
                    }
                }
            }
        }
        
        return solution;
    }
}


## Comparaison des Performances des Solveurs

Nous allons tester nos solveurs implémentés sur des grilles de Sudoku de différentes difficultés : Facile, Moyen et Difficile. Nous mesurerons également le temps de résolution et vérifierons la validité des solutions trouvées. 

Les solveurs testés sont :
- **Solveur de Satisfaction de Contraintes (CSP) avec différents DecisionBuilder**
  - Choix par défaut
  - Choix simple
  - Choix taille minimale
- **Solveur de Programmation Linéaire Mixte (MIP)**
- **Solveur de Satisfaction de Contraintes SAT (CP-SAT)**

Le test est effectué sur un ensemble de 10 Sudokus pour chaque difficulté et chaque solveur. Les résultats incluent le nombre de Sudokus résolus et le temps moyen de résolution.



In [ ]:
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;
using System.Threading;
using System.Threading.Tasks;

using Microsoft.DotNet.Interactive;

var cpSolverDefault = new OrToolsCPSolver();
var cpSolverSimple = new OrToolsCPSolver
{
    VariableSelectionStrategy = SimpleCPSolver.INT_VAR_SIMPLE,
    ValueSelectionStrategy = SimpleCPSolver.INT_VALUE_SIMPLE
};
var cpSolverMinSize = new OrToolsCPSolver
{
    VariableSelectionStrategy = SimpleCPSolver.CHOOSE_MIN_SIZE_LOWEST_MIN,
    ValueSelectionStrategy = SimpleCPSolver.ASSIGN_CENTER_VALUE
};

var mipSolverIDs = new[]
{
    "SCIP",
    // "GLOP",
    // "PDLP"
};

var mipSolverTypes = new[]
{
    LinearSolver.OptimizationProblemType.CLP_LINEAR_PROGRAMMING,
    LinearSolver.OptimizationProblemType.GLOP_LINEAR_PROGRAMMING,
    // LinearSolver.OptimizationProblemType.PDLP_LINEAR_PROGRAMMING,
    LinearSolver.OptimizationProblemType.SCIP_MIXED_INTEGER_PROGRAMMING,
    // LinearSolver.OptimizationProblemType.GLPK_MIXED_INTEGER_PROGRAMMING,
    LinearSolver.OptimizationProblemType.CBC_MIXED_INTEGER_PROGRAMMING,
    LinearSolver.OptimizationProblemType.BOP_INTEGER_PROGRAMMING,
    LinearSolver.OptimizationProblemType.SAT_INTEGER_PROGRAMMING,
    // LinearSolver.OptimizationProblemType.GUROBI_LINEAR_PROGRAMMING,
    // LinearSolver.OptimizationProblemType.GUROBI_MIXED_INTEGER_PROGRAMMING,
    // LinearSolver.OptimizationProblemType.CPLEX_LINEAR_PROGRAMMING,
    // LinearSolver.OptimizationProblemType.CPLEX_MIXED_INTEGER_PROGRAMMING,
    // LinearSolver.OptimizationProblemType.XPRESS_LINEAR_PROGRAMMING,
    // LinearSolver.OptimizationProblemType.XPRESS_MIXED_INTEGER_PROGRAMMING
};

var solvers = new List<(string Name, ISudokuSolver Solver)>
{
    ("CP Solver Default", cpSolverDefault),
    ("CP Solver Simple", cpSolverSimple),
    ("CP Solver Min Size", cpSolverMinSize),
    ("SAT Solver", new OrToolsSatSolver())
};

foreach (var solverID in mipSolverIDs)
{
    solvers.Add(($"MIP Solver {solverID}", new OrToolsMIPSolver { SolverID = solverID }));
}

foreach (var solverType in mipSolverTypes)
{
    solvers.Add(($"MIP Solver {solverType}", new OrToolsMIPSolver { OptimizationProblemType = solverType }));
}

// Utilisation des méthodes de benchmarking
var results = SudokuHelper.TestSolvers(solvers);
SudokuHelper.DisplayResults(results);


Running tests...

### Interpretation des résultats de performance

Le tableau de comparaison permet d'analyser les performances relatives des différents solveurs OR-Tools sur le problème du Sudoku.

| Aspect | Observation | Signification |
|--------|-------------|---------------|
| Solveurs CP | Performances variables selon la stratégie | Le choix du DecisionBuilder impacte significativement la résolution |
| Solveur CP-SAT | Généralement plus rapide sur problèmes difficiles | La modélisation SAT offre une meilleure propagation des contraintes |
| Solveurs MIP | Temps de résolution plus élevés | La formulation 3D binaire introduit plus de variables (729 vs 81) |

**Points clés** :
1. **CP-SAT** est généralement le plus efficace pour les Sudoku difficiles grâce à son moteur SAT moderne
2. **Le paramétrage** des solveurs CP (stratégie de sélection de variables) influence notablement les performances
3. **MIP** est moins adapté ici car la formulation binaire 3D augmente considérablement la taille du problème

> **Note technique** : Pour les problèmes de satisfaction de contraintes pures comme le Sudoku, CP-SAT est souvent préférable aux solveurs MIP.


### Conclusion Générale

Les résultats des tests de performance montrent une distinction claire entre les solveurs simples plus efficaces sur les problèmes simples et et les solveurs plus sophistiqués qui passent devant sur les problèmes plus difficiles.

Une observation clé de cette analyse est l'importance de la paramétrisation des solveurs. Les différents types de solveurs MIP et les stratégies de sélection de variables et de valeurs des solveurs CP peuvent considérablement influencer les performances. Par conséquent, il est crucial de sélectionner et de paramétrer les solveurs en fonction de la nature spécifique du problème à résoudre.

## Exercices

### Exercice 1 : Comparaison des strategies de selection de variables

Le solveur CP d'OR-Tools permet de parametrer la strategie de selection des variables (`VariableSelectionStrategy`) et des valeurs (`ValueSelectionStrategy`). Ces choix ont un impact significatif sur les performances.

**Objectif** : Comparer differentes combinaisons de strategies et mesurer leur impact sur les temps de resolution.

**Indices** :
1. Boucle imbriquee sur `varStrategies` et `valStrategies`
2. Pour chaque combinaison, creer un `OrToolsCPSolver` avec les deux strategies
3. Mesurer le temps moyen de resolution sur 3 puzzles difficiles
4. Les strategies `CHOOSE_MIN_SIZE*` selectionnent la variable avec le domaine le plus restreint (similaire a MRV) : generalement plus performantes

**Verification** : Affichez les resultats tries par temps croissant.

In [ ]:
// Exercice 1 : Comparaison des strategies de selection de variables OR-Tools
using System.Diagnostics;
using System.Linq;

var varStrategies = new (string Name, int Strategy)[]
{
    ("CHOOSE_FIRST_UNBOUND", SimpleCPSolver.CHOOSE_FIRST_UNBOUND),
    ("INT_VAR_SIMPLE", SimpleCPSolver.INT_VAR_SIMPLE),
    ("CHOOSE_MIN_SIZE_LOWEST_MIN", SimpleCPSolver.CHOOSE_MIN_SIZE_LOWEST_MIN),
    ("CHOOSE_MIN_SIZE", SimpleCPSolver.CHOOSE_MIN_SIZE)
};

var valStrategies = new (string Name, int Strategy)[]
{
    ("ASSIGN_MIN_VALUE", SimpleCPSolver.ASSIGN_MIN_VALUE),
    ("ASSIGN_MAX_VALUE", SimpleCPSolver.ASSIGN_MAX_VALUE),
    ("ASSIGN_RANDOM_VALUE", SimpleCPSolver.ASSIGN_RANDOM_VALUE),
    ("ASSIGN_CENTER_VALUE", SimpleCPSolver.ASSIGN_CENTER_VALUE)
};

var hardPuzzles3 = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).Take(3).ToList();

// TODO : Creer une boucle imbriquee sur varStrategies et valStrategies
// Pour chaque combinaison (varStrat, valStrat) :
//   var solver = new OrToolsCPSolver
//   {
//       VariableSelectionStrategy = varStrat.Strategy,
//       ValueSelectionStrategy = valStrat.Strategy
//   };
//   Mesurer le temps moyen sur hardPuzzles3 et stocker le resultat

// TODO : Afficher les resultats tries par temps croissant

throw new NotImplementedException("A vous de jouer !");

### Exercice 2 : Implémenter le Sudoku X (avec diagonales)

Le **Sudoku X** est une variante ou les deux diagonales principales doivent contenir tous les chiffres 1-9.

**Objectif** : Creer `SudokuXCPSATSolver` en ajoutant des contraintes de diagonale au solveur CP-SAT.

**Indices** :
1. Creer un `CpModel` et les variables comme dans `OrToolsSatSolver`
2. La diagonale principale : cellules `grid[i, i]` pour i de 0 a 8
3. L'anti-diagonale : cellules `grid[i, 8-i]` pour i de 0 a 8
4. Utiliser `model.AddAllDifferent()` sur chaque diagonale

> La plupart des puzzles Sudoku normaux n'ont pas de solution valide avec contraintes de diagonale. Verifiez le fonctionnement sans diagonales d'abord.

In [ ]:
// Exercice 2 : Sudoku X avec contraintes de diagonale (CP-SAT)
using Google.OrTools.Sat;

public class SudokuXCPSATSolver : ISudokuSolver
{
    private const int Dimension = 9;

    public SudokuGrid Solve(SudokuGrid inputGrid)
    {
        var model = new CpModel();
        var grid = new SatIntVar[Dimension, Dimension];

        // Creer les variables
        for (int i = 0; i < Dimension; i++)
            for (int j = 0; j < Dimension; j++)
            {
                int value = inputGrid.Cells[i, j];
                grid[i, j] = model.NewIntVar(value == 0 ? 1 : value, value == 0 ? Dimension : value, $"Cell({i},{j})");
            }

        // TODO : Ajouter les contraintes standards (lignes, colonnes, blocs 3x3)
        // Indice : model.AddAllDifferent sur chaque ligne, colonne et bloc 3x3

        // TODO : Ajouter les contraintes de diagonale
        // var mainDiag = new SatIntVar[Dimension];
        // var antiDiag = new SatIntVar[Dimension];
        // for (int i = 0; i < Dimension; i++) { mainDiag[i] = grid[i, i]; antiDiag[i] = grid[i, 8 - i]; }
        // model.AddAllDifferent(mainDiag);
        // model.AddAllDifferent(antiDiag);

        throw new NotImplementedException("A vous d'implementer le Sudoku X !");
    }
}

Console.WriteLine("TODO : Implementez SudokuXCPSATSolver avec les contraintes de diagonale");

### Exercice 3 : Comparer CSP, CP-SAT et MIP sur un puzzle difficile

Le notebook presente trois approches OR-Tools : CSP classique, CP-SAT, et MIP. Analysez leurs differences de performance.

**Objectif** : Comparer les trois solveurs sur un puzzle difficile et analyser pourquoi leurs performances different.

**Indices** :
1. Creer les trois solveurs : `OrToolsCPSolver`, `OrToolsSatSolver`, `OrToolsMIPSolver`
2. Chronometrer la resolution pour chaque solveur sur le meme puzzle difficile
3. CP-SAT est generalement le plus rapide (CDCL, apprentissage de clauses)
4. MIP est plus lent : formulation binaire 3D avec 729 variables vs 81 pour CSP/SAT

In [ ]:
// Exercice 3 : Comparaison CSP vs CP-SAT vs MIP
using System.Diagnostics;

var hardPuzzle1 = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).First();
Console.WriteLine("Puzzle difficile :");
Console.WriteLine(hardPuzzle1);
Console.WriteLine();

// TODO : Tester les trois solveurs et mesurer le temps de resolution
// var solversToCompare = new (string Name, ISudokuSolver Solver)[]
// {
//     ("CSP Classique", new OrToolsCPSolver()),
//     ("CP-SAT", new OrToolsSatSolver()),
//     ("MIP (SCIP)", new OrToolsMIPSolver { OptimizationProblemType = LinearSolver.OptimizationProblemType.SCIP_MIXED_INTEGER_PROGRAMMING })
// };
//
// Console.WriteLine($"{"Solveur",-20} | {"Temps (ms)",-12} | Statut");
// Console.WriteLine(new string('-', 45));
// foreach (var (name, solver) in solversToCompare)
// {
//     var sw = Stopwatch.StartNew();
//     var result = solver.Solve((SudokuGrid)hardPuzzle1.Clone());
//     sw.Stop();
//     Console.WriteLine($"{name,-20} | {sw.ElapsedMilliseconds,-12} | OK");
// }

Console.WriteLine("TODO : Implementez la comparaison CSP vs CP-SAT vs MIP");

---

**Navigation** : [<< Sudoku-9 Graph Coloring C#](Sudoku-9-GraphColoring-Csharp.ipynb) | [Index](README.md) | [Sudoku-11 Choco C# >>](Sudoku-11-Choco-Csharp.ipynb)

**Voir aussi** : 
- [Search-6-CSP-Fundamentals](../Search/Foundations/Search-6-CSP-Fundamentals.ipynb) - Fondamentaux CSP
- [Search-7-CSP-Consistency](../Search/Foundations/Search-7-CSP-Consistency.ipynb) - Propagation de contraintes
- [App-1-NQueens](../Search/Applications/App-1-NQueens.ipynb) - Autre application CSP

## Exercice : Solveur OR-Tools pour le probleme des N-Reines

### Enonce

Le probleme des N-Reines demande de placer N reines sur un echiquier N×N sans qu'aucune ne s'attaque. Implementez un solveur CP-SAT pour ce probleme en adaptant la modelisation du Sudoku :

1. Creez N variables entiere `queen[i]` representant la colonne de la reine de la ligne i (domaine 0..N-1)
2. Ajoutez les contraintes d'unicite pour les colonnes (AllDifferent sur `queen`)
3. Ajoutez les contraintes de diagonales : les reines ne doivent pas etre sur la meme diagonale
4. Comptez le nombre de solutions pour N=8 (attendu : 92)

### Indice

Pour les diagonales, deux reines aux positions (i, queen[i]) et (j, queen[j]) sont en conflit si `queen[i] - queen[j] == i - j` (diagonale principale) ou `queen[i] - queen[j] == j - i` (diagonale secondaire). Ces contraintes peuvent s'exprimer avec `model.Add(queen[i] - queen[j] != i - j)` pour tous les couples (i, j).

In [ ]:
// EXERCICE : Probleme des N-Reines avec OR-Tools CP-SAT
// TODO: Implementez un solveur pour compter les solutions du probleme des N-Reines

using Google.OrTools.Sat;

public class NQueensCPSATSolver
{
    public int N { get; set; }
    
    public NQueensCPSATSolver(int n)
    {
        N = n;
    }
    
    public int CountSolutions()
    {
        var model = new CpModel();
        
        // TODO: Creer les variables queen[i] pour chaque ligne i (domaine 0..N-1)
        // var queens = new IntVar[N];
        // for (int i = 0; i < N; i++)
        //     queens[i] = model.NewIntVar(0, N - 1, $"queen_{i}");
        
        // TODO: Contrainte de colonnes : toutes les reines dans des colonnes differentes
        // model.AddAllDifferent(queens);
        
        // TODO: Contraintes de diagonales
        // Pour chaque paire (i, j) avec i < j :
        //   Diagonale principale : queens[i] - queens[j] != i - j
        //   Diagonale secondaire : queens[i] - queens[j] != j - i
        
        // TODO: Utiliser un SolutionCollector pour compter les solutions
        // Classe CpSolverSolutionCallback pour enumerer toutes les solutions
        
        throw new NotImplementedException("A vous de jouer !");
    }
}

// Test de votre implementation
int N = 8;
var nQueens = new NQueensCPSATSolver(N);
// int solutions = nQueens.CountSolutions();
// Console.WriteLine($"Nombre de solutions pour {N}-Reines : {solutions}");
// Console.WriteLine($"Attendu : 92");
Console.WriteLine($"TODO: Implementez NQueensCPSATSolver.CountSolutions() pour le probleme des {N}-Reines");